# 104 — Vectorize Postprocessed Tiles

Vectorizes each postprocessed binary tile into polygons and centroids.
Applies MMU filter (1m², 2m², 5m²) per tile.
Saves per-tile GeoPackages to `vector_tiles_t{thresh}_min{label}/`.

Run after `103_postprocess.ipynb`.
Run `105_mosaic_vectors.ipynb` after this to merge all tile vectors.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config import *
from helpers import *
import time
import gc
import numpy as np

print(f"Config loaded ✓")
print(f"  Mode:       {MODE}")
print(f"  Thresholds: {PIXEL_THRESHOLDS}")
print(f"  MMU sizes:  {MMU_SIZES}")

In [ ]:
# ── Build tile list ───────────────────────────────────────────────────────────
print("── Building tile list ──────────────────────────────────────")

# Per-threshold work lists — based on what's in pp_tile_dir
to_process_per_thresh = {}
for thresh in PIXEL_THRESHOLDS:
    pp_dir    = pp_tile_dir(thresh)
    pp_files  = [f for f in os.listdir(pp_dir) if f.endswith(".tif")]
    # Convert pp filename back to source filename
    to_process_per_thresh[thresh] = [
        f.replace(f"_pp_t{thresh}.tif", ".tif") for f in pp_files
    ]
    print(f"  PP tiles t={thresh}: {len(to_process_per_thresh[thresh])}")

# Resume status
print(f"\n  Resume status:")
for thresh in PIXEL_THRESHOLDS:
    tiles = to_process_per_thresh[thresh]
    for min_m2, label in MMU_SIZES:
        vec_existing = set(os.listdir(vector_tile_dir(thresh, label)))
        done = sum(1 for f in tiles
                   if f.replace(".tif", f"_poly_t{thresh}_min{label}.gpkg") in vec_existing)
        print(f"    t={thresh} min={label}: {done}/{len(tiles)} done")

# Remaining — use ref threshold as driver
ref_thresh            = PIXEL_THRESHOLDS[0]
ref_min_m2, ref_label = MMU_SIZES[0]
vec_existing          = set(os.listdir(vector_tile_dir(ref_thresh, ref_label)))
to_process            = to_process_per_thresh[ref_thresh]
remaining             = [
    fname for fname in to_process
    if fname.replace(".tif", f"_poly_t{ref_thresh}_min{ref_label}.gpkg") not in vec_existing
]
print(f"\n  To vectorize: {len(remaining)}")

In [ ]:
# ── Vectorize per tile ────────────────────────────────────────────────────────
print("── Vectorizing tiles ───────────────────────────────────────")

t0      = time.time()
skipped = 0
total_patches = {thresh: {label: 0 for _, label in MMU_SIZES} for thresh in PIXEL_THRESHOLDS}

for i, source_fname in enumerate(remaining):
    remaining_count = len(remaining) - i - 1

    for thresh in PIXEL_THRESHOLDS:
        pp_fname = source_fname.replace(".tif", f"_pp_t{thresh}.tif")
        pp_path  = os.path.join(pp_tile_dir(thresh), pp_fname)

        if not os.path.exists(pp_path):
            continue

        # Load postprocessed tile
        with rasterio.open(pp_path) as src:
            data          = src.read(1)
            ref_transform = src.transform
            ref_crs       = src.crs
            pixel_area_m2 = abs(src.transform[0] * src.transform[4])

        if int((data == 1).sum()) == 0:
            continue

        # Vectorize all patches
        polygons = [
            {"geometry": shape(geom), "value": int(val)}
            for geom, val in shapes(data.astype(np.uint8), transform=ref_transform)
            if int(val) == 1
        ]
        del data; gc.collect()

        if not polygons:
            continue

        gdf = gpd.GeoDataFrame(polygons, crs=ref_crs)
        gdf["area_m2"] = gdf.geometry.area.round(2)
        gdf["area_ha"] = (gdf["area_m2"] / 10000).round(4)
        gdf = gdf[gdf["area_m2"] >= 1].reset_index(drop=True)

        if len(gdf) == 0:
            continue

        # Save three MMU variants
        for min_m2, label in MMU_SIZES:
            gdf_filt = gdf[gdf["area_m2"] >= min_m2].reset_index(drop=True)
            if len(gdf_filt) == 0:
                continue
            gdf_filt["patch_id"] = gdf_filt.index + 1

            # Polygons
            poly_path = os.path.join(
                vector_tile_dir(thresh, label),
                source_fname.replace(".tif", f"_poly_t{thresh}_min{label}.gpkg")
            )
            gdf_filt[["patch_id", "area_m2", "area_ha", "geometry"]].to_file(
                poly_path, driver="GPKG")

            total_patches[thresh][label] += len(gdf_filt)

        del gdf; gc.collect()

    # Progress
    elapsed = time.time() - t0
    done    = i + 1
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(remaining):
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────────")
        print(f"     Elapsed: {elapsed/3600:.1f}h  |  ETA: {eta/3600:.1f}h\n")

print(f"\n  Done. Time: {(time.time()-t0)/3600:.1f}h")
print(f"\n── Patch counts ─────────────────────────────────────────────")
for thresh in PIXEL_THRESHOLDS:
    for min_m2, label in MMU_SIZES:
        print(f"  t={thresh} min={label}: {total_patches[thresh][label]:,} patches")